# Carga de datos

In [ ]:
import os
import warnings
import logging

# Suprimir mensajes específicos de PyTorch Lightning
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PL_DISABLE_FORK_WARNING'] = '1'

warnings.filterwarnings('ignore')

logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning.accelerators.cuda").setLevel(logging.ERROR)

In [ ]:
from Modulos.Carga_de_datos import Carga_Universal_Chile

# Definimos los parámetros para la carga de datos
ruta_chile = "Datos/D"
mapa_csv = {'barra': 'Barra',
            'anio':'year',
            'mes':'month',
            'dia':'day',
            'valor': 'Valor'}

renombrar_cols = {'ATACAMA_______220': 'ATACAMA', 
                  'CARDONES_______220': 'CARDONES', 
                  'CHARRUA_______220': 'CHARRUA', 
                  'CRUCERO_______220': 'CRUCERO', 
                  'P.AZUCAR_______220': 'P.AZUCAR', 
                  'P.MONTT_______220': 'P.MONTT', 
                  'QUILLOTA_______220': 'QUILLOTA', 
                  'TARAPACA_______220': 'TARAPACA'}

eliminar_cols = ['tension']
ordenar_cols = ['Barra', 'Fecha', 'Valor']
ano_inicio = 2020
ano_fin = 2025
extension = "*.csv"
sep = ';'

df_por_barras, lista_barras = Carga_Universal_Chile(ruta_chile, mapa_csv, 
                                                renombrar_cols=renombrar_cols, 
                                                eliminar_cols=eliminar_cols, 
                                                ordenar_cols=ordenar_cols, 
                                                rango_anos=(ano_inicio, ano_fin), 
                                                extension=extension, 
                                                sep=sep)

# Preprocesamiento Prophet

In [ ]:
from Modulos.Preprocesamiento_Prophet import procesamiento_por_barras

# Parámetros para el preprocesamiento
freq = 'D'
proporciones = [0.7, 0.15, 0.15]

prepro_por_barras = procesamiento_por_barras(df_por_barras, lista_barras, freq, ano_inicio, ano_fin, proporciones, umbral_outliers=4)

In [ ]:
from Modulos.Preprocesamiento_Prophet import generar_estadisticas_todas_barras, crear_tabla_consolidada_splits

# Estadísticas por barra
estadisticas_por_barra = generar_estadisticas_todas_barras(prepro_por_barras, lista_barras)

# Prophet

In [ ]:
from Modulos.Prophet_Modular import optimizacion_prophet
import pandas as pd

# Definimos los parámetros
ruta_archivo = f"Modelos_Prophet/{freq}/params_prophet_{freq}_{ano_inicio}-{ano_fin}.json"
parametros = {'grid': {'changepoint_prior_scale': [0.03, 0.05, 0.1, 0.5, 1],
                       'seasonality_prior_scale': [0.5, 1, 1.5],
                       'seasonality_mode': ['additive', 'multiplicative'],
                       'growth': ['logistic']},
            'daily_seasonality': False,
            'weekly_seasonality': True,
            'yearly_seasonality': True,
            'techo': 1.5}
modo = 'usar'

# Buscamos los mejores hiperparámetros para cada barra
metrica = 'MAE'
mejores_params_por_barra = optimizacion_prophet(prepro_por_barras, ruta_archivo, parametros, 
                                                modo, metrica, usar_cv=True, n_splits=5, hmap=False)

# Convertimos el diccionario a DataFrame
df_params = pd.DataFrame.from_dict(mejores_params_por_barra, orient='index')

# Le ponemos nombre a la columna del índice para que se vea mejor
df_params.index.name = 'Localidad'
df_params.reset_index(inplace=True)

print("TABLA DE HIPERPARÁMETROS PROPHET:")
display(df_params)

In [ ]:
from Modulos.Prophet_Modular import multi_prophet

# Parametros base para todos los modelos Prophet
params_base = {'daily_seasonality': False,
               'weekly_seasonality': True,
               'yearly_seasonality': True}
modo = 'cargar'                             # 'entrenar' o 'cargar'
carpeta_modelos = 'Modelos_Prophet'

# Cargar modelos guardados
modelos, df_predicciones, df_metricas_prophet = multi_prophet(prepro_por_barras=prepro_por_barras, 
                                                              parametros=mejores_params_por_barra, 
                                                              params_base=params_base,
                                                              modo=modo, 
                                                              mostrar_params=False,
                                                              mostrar_graficos=False,
                                                              mostrar_componentes=False,
                                                              growth='logistic',
                                                              techo=1.5)

# Preprocesamiento Transformer

In [ ]:
from Modulos.Preprocesamiento_Transformer import generar_clima, predicciones_con_clima, incluir_features, holiday_binary
import contextlib
import os

# Configuración de hiperpárametros
batch_size = 64
max_prediction_length = 1
max_encoder_length = 90

# Guardamos la config y scalers
carpeta_modelos = f"Multi-Modelos_TFT/{freq}/LN/pred_{max_prediction_length}_{max_encoder_length}_Features_10" # agrego v2??
carpeta_logs = f"Logs_TFT/{freq}/LN/pred_{max_prediction_length}_{max_encoder_length}_Features_10"

# Nombres de los experimentos
experimento_precios = f'Multi-TFT_Precios'
experimento_residuos = f'Multi-TFT_Residuos'

# Configuración TFT
tft_config = {'lr': 0.001,
              'hidden_size': 16, # Cambiar a 16?? og 32
              'heads': 4,
              'dropout': 0.5, # Subir a 0.4 o 0.5?? og 0.3
              'cont_size': 8, # Bajar a 8?? og 16
              'patience_lr': 5}

# Early Stopping
early_stop_config = {'min_delta': 1e-4,
                     'patience': 15} # Bajar a 10?? og 20

epochs = 100

# ================================ GENERACIÓN DE DATOS CLIMÁTICOS ================================
# Coordenadas sacadas de https://www.geodatos.net/coordenadas/chile/iquique
# Ciudades sacadas de https://www.igm.cl con miradas a https://www.coordinador.cl/wp-content/uploads/2025/11/CEN_Reporte_Energetico_SEN_Nov25.pdf
Coordenadas ={'ATACAMA': (-28.57617, -70.75938),        # Vallenar
            'CARDONES': (-27.36737, -70.33219),         # Copiapó
            'CHARRUA': (-36.82699, -73.04977),          # Concepción
            'CRUCERO': (-23.65094, -70.39752),          # Antofagasta
            'P.AZUCAR': (-29.90591, -71.25014),         # La Serena
            'P.MONTT': (-41.4693, -72.94237),           # Puerto Montt
            'QUILLOTA': (-33.036, -71.62963),           # Valparaiso
            'TARAPACA': (-20.21326, -70.15027)}         # Iquique

# Fechas
fecha_inicio = df_predicciones['ds'].min().strftime('%Y-%m-%d')
fecha_fin = df_predicciones['ds'].max().strftime('%Y-%m-%d')
carpeta_clima = f"Datos/{freq}/Climaticos"

# Generamos los datos climáticos
datos_clima = generar_clima(fecha_inicio, fecha_fin, Coordenadas, carpeta_salida=carpeta_clima, freq=freq)

# Integrar las predicciones con los datos climáticos
dict_barras_clima = predicciones_con_clima(df_predicciones, datos_clima)

# ================================ GENERACIÓN DE FERIADOS Y FEATURES ================================
# Inluimos indicador de feriados
for barra in dict_barras_clima.keys():

    # Para mutear los prints dentro del loop:
    with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
        df_bin = holiday_binary(dict_barras_clima[barra], prepro_por_barras[barra]['df_feriados'])
        dict_barras_clima[barra] = df_bin

# Incluimos otros features para mejorar el desempeño del Transformer
datos_para_transformer = incluir_features(dict_barras_clima)

# ================== GENERACIÓN DE PARÁMETROS OBLIGATORIOS PARA PYTORCH FORECASTING ==================
# Agregamos los parámetros obligatorios para PyTorch Forecasting en todas las barras
for barra in datos_para_transformer.keys():
    # time_idx: Índice entero fundamental para que el TFT entienda la secuencia
    #datos_para_transformer[barra]['time_idx'] = datos_para_transformer[barra].index
    datos_para_transformer[barra]['time_idx'] = range(len(datos_para_transformer[barra]))
    
    # serie_id: Identificador de la serie. 
    datos_para_transformer[barra]["serie_id"] = barra

In [ ]:
# Asumiendo que ya importaste la función
from Modulos.Preprocesamiento_Transformer import dividir_serie_temporal

# Generamos los conjuntos train, val y test para Transformer
proporciones = [0.7, 0.1495, 0.1505] # Para que me agarre los 311 días que me agarra Prophet

datasets_transformer = {}
for barra, df in datos_para_transformer.items():

    # Aseguramos el orden cronológico
    df = df.sort_values('ds').reset_index(drop=True)

    # Recalcular time_idx después del reset_index
    df['time_idx'] = range(len(df))
    df['serie_id'] = barra

    # # Aplicamos la división temporal
    with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
        df_train, df_val, df_test = dividir_serie_temporal(df, proporciones)
    
    # Guardar en estructura organizada
    datasets_transformer[barra] = {'train': df_train,
                                   'val': df_val,
                                   'test': df_test}
    
    # Reporte rápido
    n_total = len(df)
    n_train = len(df_train)
    n_val = len(df_val)
    n_test = len(df_test)
    
    print(f"\nBarra: {barra:<5} | Train: {n_train:<5} | Val: {n_val:<4} | Test: {n_test:<4}")
    print(f"Valores: {df['y_real'].min()} a {df['y_real'].max()}")

In [ ]:
from Modulos.Preprocesamiento_Transformer import normalizar_datos, limpiar_nans

# Definimos los features y targets
#feature_cols = ['yhat', 'yhat_lower', 'yhat_upper',
#                'trend', 'yearly', 'weekly',
#                'temperatura', 'humedad', 'velocidad_viento', 'precipitacion', 'nubosidad',
#                'y_lag1', 'y_lag7', 'resid_lag1', 'resid_lag7', 
#                'rolling_mean_7', 'rolling_std_7', 'rolling_mean_resid_7', 'rolling_std_resid_7']

feature_cols = ['yhat', 'y_lag1', 'rolling_mean_7', 'y_lag7', 'resid_lag1', 'trend', 
                'rolling_mean_resid_7', 'rolling_std_7', 'rolling_std_resid_7', 
                'resid_lag7', 'yearly']

targets_cols = ['y_real', 'residuo']

# Normalizamos
datasets_norm, diccionario_scalers = normalizar_datos(datasets_transformer, feature_cols, targets_cols)

datasets_norm = limpiar_nans(lista_barras, datasets_norm)

# Validamos las escalas, rangos y continuidad de time_idx en cada barra
for barra in lista_barras:
    print(f"\n{barra}:")
    df_train = datasets_norm[barra]['train']
    
    # Escalas
    print(f"  1. Escalas:")
    print(f"     y_real: mean={df_train['y_real'].mean():>7.4f}, std={df_train['y_real'].std():>6.4f}, rango=[{df_train['y_real'].min():.2f}, {df_train['y_real'].max():.2f}]")
    print(f"     yhat:   mean={df_train['yhat'].mean():>7.4f}, std={df_train['yhat'].std():>6.4f}, rango=[{df_train['yhat'].min():.2f}, {df_train['yhat'].max():.2f}]")
    print(f"     residuo:   mean={df_train['residuo'].mean():>7.4f}, std={df_train['residuo'].std():>6.4f}, rango=[{df_train['residuo'].min():.2f}, {df_train['residuo'].max():.2f}]")

    diff_mean = abs(df_train['y_real'].mean() - df_train['yhat'].mean())
    diff_std = abs(df_train['y_real'].std() - df_train['yhat'].std())
    
    # Continuidad time_idx
    df_val = datasets_norm[barra]['val']
    df_test = datasets_norm[barra]['test']
    
    train_max = df_train['time_idx'].max()
    val_min = df_val['time_idx'].min()
    val_max = df_val['time_idx'].max()
    test_min = df_test['time_idx'].min()
    
    gap_train_val = val_min - train_max
    gap_val_test = test_min - val_max
    
    if gap_train_val == 1 and gap_val_test == 1:
        print(f"  2.  time_idx continuo entre splits")
    else:
        print(f"  2.  ERROR: Gaps en time_idx (train→val: {gap_train_val}, val→test: {gap_val_test})")
    
    print(f"  3. Tamaños: Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

In [ ]:
from Modulos.Preprocesamiento_Transformer import crear_dataloaders

# Configuración de features
#known_reals = ['yhat', 'yhat_lower', 'yhat_upper',
#               'trend', 'yearly', 'weekly',
#               'temperatura', 'humedad', 'velocidad_viento', 'precipitacion', 'nubosidad', 
#               'is_holiday', 'is_weekend',
#               'month_sin', 'month_cos', 
#               'day_week_sin', 'day_week_cos', 
#               'day_year_sin', 'day_year_cos']

known_reals = ['trend', 'yearly']

unknown_reals_price = ['y_real', 
                       'y_lag1', 'y_lag7', 
                       'rolling_mean_7', 'rolling_std_7']
                                          
unknown_reals_resid = ['residuo', 
                       'resid_lag1', 'resid_lag7', 
                       'rolling_mean_resid_7', 'rolling_std_resid_7']

dataloaders_precios, dataloaders_residuos, datasets_precios, datasets_residuos = crear_dataloaders(lista_barras, datasets_norm, 
                                                                                                   known_reals, unknown_reals_price, unknown_reals_resid,
                                                                                                   max_encoder_length, max_prediction_length, batch_size)

# Validaciones
for barra in lista_barras:
    print(f"\n{barra}:")
    
    # Verificamos que datasets_norm[barra]['test'] NO fue sobrescrito
    n_test_original = len(datasets_norm[barra]['test'])
    print(f"  Test original: {n_test_original} filas (debe ser 311)")
    
    if n_test_original > 320:
        print(f"  ADVERTENCIA: Test parece estar extendido (> 320 filas)")
    else:
        print(f"  Test tiene tamaño correcto")
    
    # Verificar batches
    n_batches_train = len(dataloaders_precios[barra]['train'])
    n_batches_val = len(dataloaders_precios[barra]['val'])
    n_batches_test = len(dataloaders_precios[barra]['test'])
    
    print(f"  Batches por época:")
    print(f"    Train: {n_batches_train}")
    print(f"    Val:   {n_batches_val}")
    print(f"    Test:  {n_batches_test}")

print(f"\nDataloaders listos\n")

In [ ]:
import os

# Diccionarios por barra
dataloaders_dict_precios = {barra: dataloaders_precios[barra] for barra in lista_barras}
dataloaders_dict_residuos = {barra: dataloaders_residuos[barra] for barra in lista_barras}

datasets_dict_precios = {barra: datasets_precios[barra]['train'] for barra in lista_barras}
datasets_dict_residuos = {barra: datasets_residuos[barra]['train'] for barra in lista_barras}

# Creamos la carpeta si no existe
os.makedirs(carpeta_modelos, exist_ok=True)
os.makedirs(carpeta_logs, exist_ok=True)

# Guardamos la configuración de features
feature_config = {'known_reals': known_reals,
                  'unknown_reals_price': unknown_reals_price,
                  'unknown_reals_resid': unknown_reals_resid,
                  'max_encoder_length': max_encoder_length,
                  'max_prediction_length': max_prediction_length,
                  'batch_size': batch_size,
                  'lista_barras': lista_barras}

# Validación preentrenamiento
print("\nVALIDACIÓN PRE-ENTRENAMIENTO")

for barra in lista_barras:
    print(f"\n{barra}:")
    
    # Verificamos el dataloader de precios
    train_loader = dataloaders_dict_precios[barra]['train']
    try:
        batch = next(iter(train_loader))
        encoder_shape = batch[0]['encoder_cont'].shape
        target_shape = batch[1][0].shape
        
        print(f"  Precios:")
        print(f"          Encoder shape: {encoder_shape}")
        print(f"          Target shape: {target_shape}")
        print(f"          Batch size: {encoder_shape[0]}")
        
        # Verificamos que no hay NaNs
        if batch[0]['encoder_cont'].isnan().any():
            print(f"      ADVERTENCIA: NaNs en encoder")
        if batch[1][0].isnan().any():
            print(f"      ADVERTENCIA: NaNs en target")
            
    except Exception as e:
        print(f"  ERROR en dataloader de precios: {e}")
        raise
    
    # Verificamos Dataloader de residuos
    train_loader_resid = dataloaders_dict_residuos[barra]['train']
    try:
        batch_resid = next(iter(train_loader_resid))
        print(f"  Residuos:")
        print(f"          Encoder shape: {batch_resid[0]['encoder_cont'].shape}")
        print(f"          Target shape: {batch_resid[1][0].shape}")
    except Exception as e:
        print(f"  ERROR en dataloader de residuos: {e}")
        raise

print(f"\nCarpeta de modelos: {carpeta_modelos}")

In [ ]:
import pickle, json, os
import json
import os

# Crear carpeta
os.makedirs(carpeta_modelos, exist_ok=True)

# 1. GUARDAR DATALOADERS (lo más importante para el cluster)
with open(f'{carpeta_modelos}/dataloaders_precios.pkl', 'wb') as f:
    pickle.dump(dataloaders_dict_precios, f)
print("✓ Dataloaders precios guardados")

with open(f'{carpeta_modelos}/dataloaders_residuos.pkl', 'wb') as f:
    pickle.dump(dataloaders_dict_residuos, f)
print("✓ Dataloaders residuos guardados")

# 2. GUARDAR SCALERS (ya lo haces, está bien)
with open(f'{carpeta_modelos}/scalers.pkl', 'wb') as f:
    pickle.dump(diccionario_scalers, f)
print("✓ Scalers guardados")

# 3. GUARDAR CONFIGURACIÓN (ya lo haces, está bien)
with open(f'{carpeta_modelos}/feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)
print("✓ Feature config guardado")

# 4. GUARDAR CONFIGURACIÓN TFT
tft_config_total = {
    'tft_config': tft_config,
    'early_stop_config': early_stop_config,
    'epochs': epochs,
    'max_encoder_length': max_encoder_length,
    'max_prediction_length': max_prediction_length,
    'batch_size': batch_size,
    'freq': freq
}

with open(f'{carpeta_modelos}/tft_config.json', 'w') as f:
    json.dump(tft_config_total, f, indent=2)
print("✓ TFT config guardado")

In [ ]:
def inspeccionar_datasets_norm(datasets_norm, barra, conjunto='train'):
    """
    Inspecciona qué features tiene datasets_norm
    """
    print(f"\n{'='*80}")
    print(f"INSPECCIÓN: datasets_norm['{barra}']['{conjunto}']")
    print(f"{'='*80}\n")
    
    df = datasets_norm[barra][conjunto]
    
    print(f"Shape: {df.shape}")
    print(f"Filas: {len(df)}, Columnas: {len(df.columns)}\n")
    
    print(f"TODAS LAS COLUMNAS ({len(df.columns)}):")
    print("="*80)
    for i, col in enumerate(df.columns, 1):
        dtype = df[col].dtype
        mean_val = df[col].mean() if df[col].dtype in ['float64', 'float32'] else 'N/A'
        print(f"{i:3d}. {col:30s} | dtype: {str(dtype):15s} | mean: {mean_val}")
    
    print(f"\n\nRESUMEN POR TIPO:")
    print("="*80)
    print(f"Datetime: {[c for c in df.columns if 'ds' in c]}")
    print(f"Targets: {[c for c in df.columns if c in ['y_real', 'residuo']]}")
    print(f"Yhat (Prophet): {[c for c in df.columns if 'yhat' in c]}")
    print(f"Meta: {[c for c in df.columns if c in ['time_idx', 'serie_id', 'Barra', 'Conjunto']]}")
    print(f"Features temporales: {[c for c in df.columns if 'sin' in c or 'cos' in c]}")
    print(f"Lags: {[c for c in df.columns if 'lag' in c]}")
    print(f"Rolling: {[c for c in df.columns if 'rolling' in c]}")
    
    # Verificaciones
    print(f"\n\nVERIFICACIONES:")
    print("="*80)
    
    if 'yhat' in df.columns:
        print(f"✅ yhat PRESENTE")
        print(f"   Mean: {df['yhat'].mean():.4f}, Std: {df['yhat'].std():.4f}")
        print(f"   Rango: [{df['yhat'].min():.2f}, {df['yhat'].max():.2f}]")
    else:
        print(f"❌ yhat AUSENTE")
    
    if 'y_real' in df.columns:
        print(f"✅ y_real PRESENTE")
        print(f"   Mean: {df['y_real'].mean():.4f}, Std: {df['y_real'].std():.4f}")
    
    if df.isnull().sum().sum() == 0:
        print(f"✅ NO hay NaN")
    else:
        print(f"❌ Hay NaN:")
        print(df.isnull().sum()[df.isnull().sum() > 0])

print("PASO 1: Ver qué hay en datasets_norm")
print("="*80)

inspeccionar_datasets_norm(datasets_norm, 'ATACAMA', 'train')


In [ ]:
import pickle

# Intenta cargar los dataloaders
try:
    with open('Multi-Modelos_TFT/D/LN/pred_1_90_Features_10/dataloaders_precios.pkl', 'rb') as f:
        test_dl = pickle.load(f)
    print("✓ dataloaders_precios.pkl cargado correctamente")
    print(f"  Barras: {list(test_dl.keys())}")
except Exception as e:
    print(f"✗ ERROR al cargar dataloaders_precios.pkl: {e}")

try:
    with open('Multi-Modelos_TFT/D/LN/pred_1_90_Features_10/dataloaders_residuos.pkl', 'rb') as f:
        test_dl = pickle.load(f)
    print("✓ dataloaders_residuos.pkl cargado correctamente")
except Exception as e:
    print(f"✗ ERROR al cargar dataloaders_residuos.pkl: {e}")

# Verifica los otros archivos también
import json
with open('Multi-Modelos_TFT/D/LN/pred_1_90_Features_10/feature_config.json', 'r') as f:
    print("✓ feature_config.json cargado correctamente")

with open('Multi-Modelos_TFT/D/LN/pred_1_90_Features_10/tft_config.json', 'r') as f:
    print("✓ tft_config.json cargado correctamente")

# Entrenamiento Modelo TFT LN

In [ ]:
from Modulos.TFT_Model import entrenar_multiple_tft
import time

# Entrenamiento modelo de precios
tiempo_inicio_precios = time.time()
modelos_precios = entrenar_multiple_tft(datasets_dict_precios, dataloaders_dict_precios, 
                                        ['ATACAMA', 'CARDONES', 'CHARRUA', 'CRUCERO'],
                                        tft_config, early_stop_config,
                                        nombre_experimento=experimento_precios,
                                        epochs=epochs,
                                        carpeta_modelos=carpeta_modelos,
                                        carpeta_logs=carpeta_logs,
                                        verbose=True)
tiempo_fin_precios = time.time()
duracion_precios = (tiempo_fin_precios - tiempo_inicio_precios) / 60
print(f"ENTRENAMIENTO MODELO DE PRECIOS | DURACIÓN: {duracion_precios:.2f} minutos")

# Entrenamiento modelo de residuos
tiempo_inicio_residuos = time.time()
modelos_residuos = entrenar_multiple_tft(datasets_dict_residuos, dataloaders_dict_residuos,
                                         ['ATACAMA', 'CARDONES', 'CHARRUA', 'CRUCERO'],
                                         tft_config, early_stop_config,
                                         nombre_experimento=experimento_residuos,
                                         epochs=epochs,
                                         carpeta_modelos=carpeta_modelos,
                                         carpeta_logs=carpeta_logs,
                                         verbose=True)
tiempo_fin_residuos = time.time()
duracion_residuos = (tiempo_fin_residuos - tiempo_inicio_residuos) / 60
print(f"ENTRENAMIENTO MODELO DE RESIDUOS | DURACIÓN: {duracion_residuos:.2f} minutos")

# RESUMEN FINAL
print("\n" + "="*80)
print("ENTRENAMIENTO COMPLETADO")
print("="*80)

#duracion_total = duracion_precios + duracion_residuos
duracion_total = duracion_residuos

print(f"\nRESUMEN:")
print(f"  Barras entrenadas: {len(['ATACAMA', 'CARDONES', 'CHARRUA', 'CRUCERO'])}")
#print(f"  Modelos de precios: {len(modelos_precios)}")
print(f"  Modelos de residuos: {len(modelos_residuos)}")
print(f"\n  TIEMPOS:")
#print(f"  Precios: {duracion_precios:.2f} min")
print(f"  Residuos: {duracion_residuos:.2f} min")
print(f"  Total: {duracion_total:.2f} min ({duracion_total/60:.2f} horas)")
print(f"\nARCHIVOS GUARDADOS:")
print(f"  Modelos: {carpeta_modelos}/")
print(f"  Scalers: {carpeta_modelos}/scalers.pkl")
print(f"  Config: {carpeta_modelos}/feature_config.json")

# Carga de modelos entrenados

In [ ]:
from Modulos.TFT_Model import cargar_modelo_entrenado
# El ultimo que entrené tardó 128 min para 2 barras y 2 modelos

modelos_precios = {}
configs_precios = {}
modelos_residuos = {}
configs_residuos = {}
b = ['ATACAMA', 'CARDONES', 'CHARRUA', 'CRUCERO']
for barra in b:
    modelo_precios, config_precios = cargar_modelo_entrenado(barra, experimento_precios, carpeta_modelos=carpeta_modelos)
    modelos_precios[barra] = modelo_precios
    configs_precios[barra] = config_precios

    modelo_residuos, config_residuos = cargar_modelo_entrenado(barra, experimento_residuos, carpeta_modelos=carpeta_modelos)
    modelos_residuos[barra] = modelo_residuos
    configs_residuos[barra] = config_residuos

In [ ]:
from Modulos.TFT_Model import grafico_losses

grafico_losses(lista_barras=lista_barras,
               logs_dir=carpeta_logs,
               experimento_precios=experimento_precios,
               experimento_residuos=experimento_residuos,
               nombre_modelo='TFT_LN')

# Analisis de importancia


In [ ]:
from Modulos.Feature_Analysis import analizar_importancia_features

# USO SIMPLE:
resultados_importancia = analizar_importancia_features(
    datasets_norm=datasets_norm,
    feature_cols=feature_cols,
    target_col='y_real',
    barras=['ATACAMA', 'CARDONES', 'CHARRUA', 'CRUCERO'],
    carpeta_logs=carpeta_logs,
    verbose=True
)

# Acceder a resultados
ranking = resultados_importancia['ranking_consolidado']
correlaciones = resultados_importancia['corr_promedio']
vif = resultados_importancia['vif_promedio']

# Usar para siguiente análisis
top_10_features = ranking.head(10).index.tolist()
print(f"TOP 10 features: {top_10_features}")

In [ ]:
from Modulos.Feature_Analysis import analizar_importancia_features

# USO SIMPLE:
resultados_importancia = analizar_importancia_features(
    datasets_norm=datasets_norm,
    feature_cols=feature_cols,
    target_col='residuo',
    barras=['ATACAMA', 'CARDONES', 'CHARRUA', 'CRUCERO'],
    carpeta_logs=carpeta_logs,
    verbose=True
)

# Acceder a resultados
ranking = resultados_importancia['ranking_consolidado']
correlaciones = resultados_importancia['corr_promedio']
vif = resultados_importancia['vif_promedio']

# Usar para siguiente análisis
top_10_features = ranking.head(10).index.tolist()
print(f"TOP 10 features: {top_10_features}")

# Analisis de desfase

In [ ]:
from Modulos.Desfase import analisis_de_desfase
import random

# Seleccionar una barra aleatoriamente
barra_aleatoria = random.choice(lista_barras)
print(f"Barra seleccionada aleatoriamente: {barra_aleatoria}")
barra_aleatoria = ['CARDONES']
# Hacemos el analisis, todas deberían tener el mismo desfase
resultados_desfase_ln = analisis_de_desfase(lista_barras=barra_aleatoria,
                                            modelos_precios=modelos_precios,
                                            dataloaders_precios=dataloaders_precios,
                                            datasets_norm=datasets_norm,
                                            diccionario_scalers=diccionario_scalers,
                                            max_encoder_length=max_encoder_length,
                                            max_prediction_length=max_prediction_length,
                                            horizonte=1,
                                            nombre_modelo='TFT_LN',
                                            rango_desfase=(-(max_encoder_length + 20), max_encoder_length + 21),
                                            n_puntos_plot=200,
                                            guardar_graficos=False,
                                            mostrar_graficos=True,
                                            verbose=True)

# Evaluación

In [ ]:
from Modulos.Evaluacion_TFT import evaluar_modelos_tft

# Obtener desfase de analisis_de_desfase
desfase = resultados_desfase_ln[barra_aleatoria[0]]['desfase_optimo']

datos_ln, metricas_ln = evaluar_modelos_tft(lista_barras=b,
                                            modelos_precios=modelos_precios,
                                            modelos_residuos=modelos_residuos,
                                            dataloaders_precios=dataloaders_precios,
                                            dataloaders_residuos=dataloaders_residuos,
                                            datasets_norm=datasets_norm,
                                            diccionario_scalers=diccionario_scalers,
                                            desfase=desfase,
                                            horizonte=1,
                                            nombre_modelo='TFT_LN',
                                            umbral_desfase=0.001,
                                            guardar_graficos=False,
                                            mostrar_graficos=True,
                                            verbose=False)

# Stacking Optimization

In [ ]:
from Modulos.Stacking_Optimization_TFT import stacking_optimization

# Colores personalizados para las gráficas
mis_colores = ["#eb311c", "#e2ec1a", "#1bdd6bff", "#000000", "#2812f3", "#b300ff",]

resultados_stacking_ln, df_stacking_ln = stacking_optimization(lista_barras=b,
                                                               modelos_precios=modelos_precios, 
                                                               modelos_residuos=modelos_residuos,
                                                               dataloaders_precios=dataloaders_precios, 
                                                               dataloaders_residuos=dataloaders_residuos,
                                                               datasets_norm=datasets_norm,
                                                               diccionario_scalers=diccionario_scalers,
                                                               nombre_modelo='TFT_LN',
                                                               grid_resolution=41,
                                                               metrica_optimizacion='MAE',
                                                               colormap=None,
                                                               colores_personalizados=mis_colores)

In [ ]:
from Modulos.Stacking_Optimization_TFT import comparar_metodos_stacking

comparacion_ln, df_metodos_ln = comparar_metodos_stacking(lista_barras=b,
                                                          resultados_stacking=resultados_stacking_ln,
                                                          nombre_modelo='TFT_LN')

In [ ]:
from Modulos.Stacking_Optimization_TFT import graficar_stacking

# Solo graficar 2 estrategias con colores específicos
colores_custom = {'Prophet_Residuos_Precios': '#E74C3C',  # Rojo
                  'TFT_Precios_Solo': '#3498DB'           # Azul
}

# Solo grafica la mejor estrategia de cada barra
resultados_graficos_ln = graficar_stacking(resultados_stacking=resultados_stacking_ln,
                                           lista_barras=b,
                                           nombre_modelo='TFT_LN',
                                           estrategias_plot=None, 
                                           metrica_mejor='MAE',
                                           conjunto_mejor='test',
                                           guardar_graficos=False,
                                           verbose=True)

df_resumen_ln = resultados_graficos_ln['df_resumen']
df_mejores_ln = resultados_graficos_ln['df_mejores']

# Entrenamiento Modelo TFT DyT

In [ ]:
from Modulos.Parche import crear_tft_con_dyt, count_layers
from lightning.pytorch import LightningModule

carpeta_modelos_dyt = f"Multi-Modelos_TFT/{freq}/DyT/pred_{max_prediction_length}_{max_encoder_length}_Features_Prophet"
carpeta_logs_dyt = f"Logs_TFT/{freq}/DyT/pred_{max_prediction_length}_{max_encoder_length}_Features_Prophet"
experimento_precios_dyt = f'Multi-TFT_DyT_Precios_{freq}'
experimento_residuos_dyt = f'Multi-TFT_DyT_Residuos_{freq}'

modelos_precios_dyt = {}
modelos_residuos_dyt = {}

for barra in b:
    
    # MODELO DE PRECIOS
    modelo_precios_dyt = crear_tft_con_dyt(datasets_precios[barra]['train'], tft_config, verbose=False)
    modelos_precios_dyt[barra] = modelo_precios_dyt
    
    # MODELO DE RESIDUOS
    modelo_residuos_dyt = crear_tft_con_dyt(datasets_residuos[barra]['train'], tft_config, verbose=False)
    modelos_residuos_dyt[barra] = modelo_residuos_dyt

for barra in b:
    print(f"{barra}:")
    
    for tipo, modelos in [('Precios', modelos_precios_dyt), ('Residuos', modelos_residuos_dyt)]:
        modelo = modelos[barra]
        is_lightning = isinstance(modelo, LightningModule)
        ln_count, dyt_count = count_layers(modelo)
        params = sum(p.numel() for p in modelo.parameters())
        
        status = "LISTO" if (is_lightning and ln_count == 0 and dyt_count > 0) else "⚠️"
        
        print(f"   {status} {tipo:10s}: LightningModule={is_lightning}, LN={ln_count:2d}, DyT={dyt_count:3d}, Params={params:,}")

In [ ]:
from datetime import datetime
from Modulos.DyT import entrenar_y_guardar_tft_dyt
import json

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Crear carpetas
os.makedirs(carpeta_modelos_dyt, exist_ok=True)
os.makedirs(carpeta_logs_dyt, exist_ok=True)

print("\n" + "="*80)
print("CONFIGURACIÓN DE ENTRENAMIENTO DyT")
print("="*80)
print(f"   Carpeta base: {carpeta_modelos_dyt}/")
print(f"   Carpeta logs: {carpeta_logs_dyt}/")
print(f"   Timestamp:    {timestamp}")
print("="*80 + "\n")

resultados_entrenamiento = {}

for barra in ['ATACAMA', 'CARDONES', 'CHARRUA', 'CRUCERO']:
    
    print(f"\n{'='*80}")
    print(f"BARRA: {barra}")
    print(f"{'='*80}\n")

    resultados_entrenamiento[barra] = {}

    print(f"\nMODELO DE PRECIOS")
    
    modelo_p, trainer_p, best_p = entrenar_y_guardar_tft_dyt(
        modelo=modelos_precios_dyt[barra],
        train_dl=dataloaders_precios[barra]['train'],
        val_dl=dataloaders_precios[barra]['val'],
        barra=barra,
        tipo_modelo='Precios',
        tft_config=tft_config,
        early_stop_config=early_stop_config,
        epochs=epochs,
        timestamp=timestamp,
        carpeta_modelos=carpeta_modelos_dyt,
        carpeta_logs=carpeta_logs_dyt)
    
    resultados_entrenamiento[barra]['Precios'] = {'modelo': modelo_p,
                                                  'trainer': trainer_p,
                                                  'best_path': best_p}

    print(f"\nMODELO DE RESIDUOS")
    
    modelo_r, trainer_r, best_r = entrenar_y_guardar_tft_dyt(
        modelo=modelos_residuos_dyt[barra],
        train_dl=dataloaders_residuos[barra]['train'],
        val_dl=dataloaders_residuos[barra]['val'],
        barra=barra,
        tipo_modelo='Residuos',
        tft_config=tft_config,
        early_stop_config=early_stop_config,
        epochs=epochs,
        timestamp=timestamp,
        carpeta_modelos=carpeta_modelos_dyt,
        carpeta_logs=carpeta_logs_dyt)
    
    resultados_entrenamiento[barra]['Residuos'] = {'modelo': modelo_r,
                                                   'trainer': trainer_r,
                                                   'best_path': best_r}

print("\n" + "="*80)
print("RESUMEN DE ENTRENAMIENTO")
print("="*80 + "\n")

for barra in lista_barras:
    print(f"{barra}:")
    
    for tipo in ['Precios', 'Residuos']:
        # Cargar config del archivo JSON
        experimento = f"DyT_{tipo}"
        config_path = os.path.join(carpeta_modelos_dyt, experimento, barra, 
                                   f"{experimento}_{barra}_config.json")
        
        with open(config_path, 'r', encoding='utf-8') as f:
            config = json.load(f)
        
        print(f"   {tipo:10s}: Epochs={config['epochs_trained']:3d} | "
              f"Val Loss={config['best_val_loss']:.6f}")
    print()

print("="*80 + "\n")

# ESTRUCTURA DE ARCHIVOS
print(f"📂 Estructura creada en: {carpeta_modelos_dyt}/")
print("\n📁 Estructura de carpetas:")
print(f"   {carpeta_modelos_dyt}/")
print(f"   ├── DyT_Precios/")
for barra in lista_barras:
    print(f"   │   ├── {barra}/")
    print(f"   │   │   ├── DyT_Precios_{barra}_best.ckpt")
    print(f"   │   │   └── DyT_Precios_{barra}_config.json")
print(f"   └── DyT_Residuos/")
for barra in lista_barras:
    print(f"       ├── {barra}/")
    print(f"       │   ├── DyT_Residuos_{barra}_best.ckpt")
    print(f"       │   └── DyT_Residuos_{barra}_config.json")

print("\n" + "="*80)
print("✅ ENTRENAMIENTO COMPLETADO")
print("="*80 + "\n")

# Cargar modelos DyT

In [ ]:
from Modulos.Parche import activar_dyt_mode, desactivar_dyt_mode
from Modulos.TFT_Model import cargar_modelo_entrenado

# Configuración
carpeta_modelos_dyt = f"Multi-Modelos_TFT/{freq}/DyT/pred_{max_prediction_length}_{max_encoder_length}_Features_Prophet"
carpeta_logs_dyt = f"Logs_TFT/{freq}/DyT/pred_{max_prediction_length}_{max_encoder_length}_Features_Prophet"
experimento_precios_dyt = f'DyT_Precios'
experimento_residuos_dyt = f'DyT_Residuos'

# Diccionarios para almacenar modelos
modelos_precios_dyt = {}
configs_precios_dyt = {}
modelos_residuos_dyt = {}
configs_residuos_dyt = {}

# Activamos el mode DyT
activar_dyt_mode()

try:
    for barra in b:
        
        # PRECIOS
        modelo_p, config_p = cargar_modelo_entrenado(barra=barra, 
                                                     nombre_experimento=experimento_precios_dyt, 
                                                     carpeta_modelos=carpeta_modelos_dyt)
        modelos_precios_dyt[barra] = modelo_p
        configs_precios_dyt[barra] = config_p

        # RESIDUOS
        modelo_r, config_r = cargar_modelo_entrenado(barra=barra, 
                                                     nombre_experimento=experimento_residuos_dyt, 
                                                     carpeta_modelos=carpeta_modelos_dyt)
        modelos_residuos_dyt[barra] = modelo_r
        configs_residuos_dyt[barra] = config_r
        
finally:
    # Desactivamos el modo DyT
    desactivar_dyt_mode()

In [ ]:
from Modulos.TFT_Model import grafico_losses

# Configuración actualizada

grafico_losses(b,
               carpeta_logs_dyt, 
               experimento_precios=experimento_precios_dyt, 
               experimento_residuos=experimento_residuos_dyt)

# Analisis de desfase

In [ ]:
# Seleccionar una barra aleatoriamente
barra_aleatoria = random.choice(lista_barras)
print(f"Barra seleccionada aleatoriamente: {barra_aleatoria}")
barra_aleatoria = ['CARDONES']
# Hacemos el analisis, todas deberían tener el mismo desfase
resultados_desfase_dyt = analisis_de_desfase(lista_barras=barra_aleatoria,
                                             modelos_precios=modelos_precios_dyt,
                                             dataloaders_precios=dataloaders_precios,
                                             datasets_norm=datasets_norm,
                                             diccionario_scalers=diccionario_scalers,
                                             max_encoder_length=max_encoder_length,
                                             max_prediction_length=max_prediction_length,
                                             horizonte=1,
                                             nombre_modelo='TFT_DyT',
                                             rango_desfase=(-(max_encoder_length + 20), max_encoder_length + 21),
                                             n_puntos_plot=200,
                                             guardar_graficos=False,
                                             mostrar_graficos=True,
                                             verbose=True)

# Evaluación

In [ ]:
# Evaluamos los modelos TFT_DyT
desfase = resultados_desfase_ln[barra_aleatoria[0]]['desfase_optimo']
b = lista_barras[:4]
datos_dyt, metricas_dyt = evaluar_modelos_tft(lista_barras=b,
                                              modelos_precios=modelos_precios_dyt, 
                                              modelos_residuos=modelos_residuos_dyt,
                                              dataloaders_precios=dataloaders_precios, 
                                              dataloaders_residuos=dataloaders_residuos,
                                              datasets_norm=datasets_norm,
                                              diccionario_scalers=diccionario_scalers,
                                              desfase=desfase,
                                              horizonte=1,
                                              nombre_modelo='TFT_DyT',
                                              umbral_desfase=0.001,
                                              guardar_graficos=False,
                                              mostrar_graficos=True,
                                              verbose=False)

# Stacking Optimization DyT

In [ ]:
# Colores personalizados para las gráficas
mis_colores = ["#eb311c", "#e2ec1a", "#1bdd6bff", "#000000", "#2812f3", "#b300ff",]

resultados_stacking_dyt, df_stacking_dyt = stacking_optimization(lista_barras=b,
                                                                 modelos_precios=modelos_precios_dyt,
                                                                 modelos_residuos=modelos_residuos_dyt,
                                                                 dataloaders_precios=dataloaders_precios,
                                                                 dataloaders_residuos=dataloaders_residuos,
                                                                 datasets_norm=datasets_norm,
                                                                 diccionario_scalers=diccionario_scalers,
                                                                 nombre_modelo='TFT_DyT',
                                                                 grid_resolution=41,
                                                                 metrica_optimizacion='MAE',
                                                                 colormap=None,
                                                                 colores_personalizados=mis_colores)

In [ ]:
comparacion_dyt, df_metodos_dyt = comparar_metodos_stacking(lista_barras=b,
                                                          resultados_stacking=resultados_stacking_dyt,
                                                          nombre_modelo='TFT_DyT')

In [ ]:
# Solo graficar 2 estrategias con colores específicos
colores_custom = {'Prophet_Residuos_Precios': '#E74C3C',  # Rojo
                  'TFT_Precios_Solo': '#3498DB'           # Azul
}

resultados_graficos_dyt = graficar_stacking(resultados_stacking=resultados_stacking_dyt,
                                           lista_barras=b,
                                           nombre_modelo='TFT_DyT',
                                           estrategias_plot=None,
                                           metrica_mejor='MAE',
                                           conjunto_mejor='test',
                                           guardar_graficos=False,
                                           verbose=True)

df_resumen_dyt = resultados_graficos_dyt['df_resumen']
df_mejores_dyt = resultados_graficos_dyt['df_mejores']

# Comparativa

In [ ]:
from Modulos.Comparativa import analisis_comparativo_completo

resultados_comparacion = analisis_comparativo_completo(
    metricas_ln=metricas_ln,
    metricas_dyt=metricas_dyt,
    df_mejores_ln=df_mejores_ln,
    df_mejores_dyt=df_mejores_dyt,
    lista_barras=b,
    comparar='Test',
    guardar_resultados=False,
    mostrar_graficos=True,
    verbose=True
)

In [ ]:
from Modulos.Comparativa import diagnostico_stacking_failure

# Ejecutar diagnóstico completo
resultados_diagnostico = diagnostico_stacking_failure(
    lista_barras=b,
    datos_ln=datos_ln,
    datos_dyt=datos_dyt,
    resultados_stacking_ln=resultados_stacking_ln,
    resultados_stacking_dyt=resultados_stacking_dyt,
    datasets_norm=datasets_norm,
    diccionario_scalers=diccionario_scalers,
    guardar_resultados=False,
    mostrar_graficos=True,
    verbose=True
)

# Acceder a diagnósticos específicos
df_correlaciones = resultados_diagnostico['diversidad']['df_correlaciones']
df_complementariedad = resultados_diagnostico['complementariedad']['df_complementariedad']
df_pesos = resultados_diagnostico['pesos']['df_pesos']

# Split Conformal Prediction

In [ ]:
from Modulos.Conformal_Prediction_Wrapper import cp_clasic_implementation, split_cp_from_calibration

# Split CP: La idea es que, despues de entrenar el modelo, se calculan los residuos en el conjunto de validación, 
# y se usa ese conjunto para calibrar el CP. Usando un cuantil de residuos q, se generan intervalos de la forma [y - q, y + q].
# Luego, se evalúa el CP en el conjunto de test para ver su desempeño real, no cambia el cuantil. 
# Intervalo fijo, ancho uniforme y no adaptativo.

cp_ln, df_cp_ln = cp_clasic_implementation(resultados_stacking=resultados_stacking_ln,
                                                   df_mejores=df_mejores_ln,
                                                   split_cp_fn=split_cp_from_calibration,
                                                   mapping='LN',
                                                   verbose=False)

cp_dyt, df_cp_dyt = cp_clasic_implementation(resultados_stacking=resultados_stacking_dyt,
                                                   df_mejores=df_mejores_dyt,
                                                   split_cp_fn=split_cp_from_calibration,
                                                   mapping='DyT',
                                                   verbose=False)
display(df_cp_ln)
display(df_cp_dyt)

In [ ]:
from Modulos.Conformal_Prediction_Wrapper import plot_cp

# Graficamos
plot_cp(['ATACAMA', 'CARDONES', 'CHARRUA', 'CRUCERO'],
        cp_ln=cp_ln,
        cp_dyt=cp_dyt,
        resultados_stacking_ln=resultados_stacking_ln,
        resultados_stacking_dyt=resultados_stacking_dyt,
        tipo_cp='Clásico') 

# Conformal Prediction EnbPI

In [ ]:
from Modulos.Conformal_Prediction_Wrapper import cp_enbpi_implementation

# EnbPI: Ensemble Bootstrap Prediction Intervals es otro método de CP que utiliza técnicas de ensemble
# y bootstrap para generar mejores intervalos de predicción, la ventaja es que son más robustos y adaptativos. 
# Entrena un RF (en este caso 100 arboles),promedia las predicciones, calcula los residuos, y luego usa bootstrap para generar 
# muestras de residuos, y luego calibra los inervalos. A diferencia del SCP, que se basa en residuos de un solo modelo, EnbPI aprovecha 
# la diversidad de múltiples modelos y muestras bootstrap para calibrar intervalos que pueden capturar mejor la 
# incertidumbre inherente a los datos y al modelo.

for e in ['RF', 'GB', 'Ridge']:
    print(f"\n🔍 Evaluando modelo base: {e}")
    # LN
    estrategia_barras_ln = df_mejores_ln.set_index('Barra')['Mejor Estrategia'].to_dict()
    df_enbpi_ln, cp_enbpi_ln = cp_enbpi_implementation(resultados_stacking_ln,
                                                       b,
                                                       estrategia_barras_ln,
                                                       modelo_base=e,
                                                       alpha=0.05,
                                                       B=30)

    print("\n📊 RESULTADOS EnbPI LN:")
    display(df_enbpi_ln)

    # DyT
    estrategia_barras_dyt = df_mejores_dyt.set_index('Barra')['Mejor Estrategia'].to_dict()
    df_enbpi_dyt, cp_enbpi_dyt = cp_enbpi_implementation(resultados_stacking_dyt,
                                                         b,
                                                         estrategia_barras_dyt,
                                                         modelo_base=e,
                                                         alpha=0.05,
                                                         B=30)

    print("\n📊 RESULTADOS EnbPI DyT:")
    display(df_enbpi_dyt)

    # Comparar LN vs DyT
    print("\n📈 COMPARACIÓN:")
    comparison = pd.DataFrame({
        "Barra": df_enbpi_ln["Barra"],
        "Coverage_LN": df_enbpi_ln["Coverage"],
        "Coverage_DyT": df_enbpi_dyt["Coverage"],
        "Width_LN": df_enbpi_ln["Width"],
        "Width_DyT": df_enbpi_dyt["Width"],
        "MAE_LN": df_enbpi_ln["MAE"],
        "MAE_DyT": df_enbpi_dyt["MAE"   ]
})
    display(comparison)

In [ ]:
# LN
estrategia_barras_ln = df_mejores_ln.set_index('Barra')['Mejor Estrategia'].to_dict()
df_enbpi_ln, cp_enbpi_ln = cp_enbpi_implementation(resultados_stacking_ln, 
                                                   b, 
                                                   estrategia_barras_ln,
                                                   modelo_base='Ridge',  # Cambiar a 'GB' o 'Ridge' si quieres
                                                   alpha=0.05,
                                                   B=30)

print("\nRESULTADOS EnbPI LN:")
display(df_enbpi_ln)

# DyT
estrategia_barras_dyt = df_mejores_dyt.set_index('Barra')['Mejor Estrategia'].to_dict()
df_enbpi_dyt, cp_enbpi_dyt = cp_enbpi_implementation(resultados_stacking_dyt, 
                                                     b, 
                                                     estrategia_barras_dyt,
                                                     modelo_base='Ridge',
                                                     alpha=0.05,
                                                     B=30)

print("\nRESULTADOS EnbPI DyT:")
display(df_enbpi_dyt)

# Comparar LN vs DyT
print("\n📈 COMPARACIÓN:")
comparison = pd.DataFrame({
    "Barra": df_enbpi_ln["Barra"],
    "Coverage_LN": df_enbpi_ln["Coverage"],
    "Coverage_DyT": df_enbpi_dyt["Coverage"],
    "Width_LN": df_enbpi_ln["Width"],
    "Width_DyT": df_enbpi_dyt["Width"],
    "MAE_LN": df_enbpi_ln["MAE"],
    "MAE_DyT": df_enbpi_dyt["MAE"]
})
display(comparison)

In [ ]:
# Graficamos
plot_cp(b, 
        cp_enbpi_ln, 
        cp_enbpi_dyt, 
        resultados_stacking_ln, 
        resultados_stacking_dyt, 
        tipo_cp='EnbPI')

# ACP Offline

In [ ]:
from Modulos.Conformal_Prediction_Wrapper import acp_offline_implementation

# Metodo de CP adaptativo, significa que cambia alpha cambia según los errores.
# Entrena el modelo una sola vez, calcula los residuos una vez y va adaptando el alpha 
# para cada predicción según la dificultad de predecir esa instancia.
# Adaptativa:
# Para cada punto del test t = 1, 2, 3,... 
# Calcular Q1, dar la predicción, observar el valor real, calcular el error, 
# actualizar alpha según el error, si el error es alto, aumentar alpha, si es bajo, 
# y luego generar el intervalo de predicción usando el nuevo alpha.  

# Probar con diferentes valores de gamma
gammas_to_test = [0.01, 0.05, 0.1]

resultados_acp = {}

for g in gammas_to_test:
    print(f"\n{'='*60}")
    print(f"🔍 EVALUANDO ACP ONLINE CON GAMMA = {g}")
    print(f"{'='*60}")
    
    # LN
    print(f"\nACP LN (gamma={g}):")
    estrategia_barras_ln = df_mejores_ln.set_index('Barra')['Mejor Estrategia'].to_dict()
    df_acp_off_ln, cp_acp_off_ln, alphas_acp_off_ln = acp_offline_implementation(resultados_stacking_ln, 
                                                                    b,
                                                                    estrategia_barras_ln,
                                                                    modelo_base='Ridge',
                                                                    alpha=0.05,
                                                                    gamma=g)
    display(df_acp_off_ln[['Barra', 'Estrategia', 'Coverage', 'Expected', 'Coverage_Diff', 'Width', 'MAE', 'RMSE', 'Alpha_Mean']])
    
    # DyT
    print(f"\nACP DyT (gamma={g}):")
    estrategia_barras_dyt = df_mejores_dyt.set_index('Barra')['Mejor Estrategia'].to_dict()
    df_acp_off_dyt, cp_acp_off_dyt, alphas_acp_off_dyt = acp_offline_implementation(resultados_stacking_dyt,
                                                                       b,
                                                                       estrategia_barras_dyt,
                                                                       modelo_base='Ridge',
                                                                       alpha=0.05,
                                                                       gamma=g)
    display(df_acp_off_dyt[['Barra', 'Estrategia', 'Coverage', 'Expected', 'Coverage_Diff', 'Width', 'MAE', 'RMSE', 'Alpha_Mean']])
    
    # Guardar resultados
    resultados_acp[g] = {'ln': (df_acp_off_ln, cp_acp_off_ln, alphas_acp_off_ln), 
                         'dyt': (df_acp_off_dyt, cp_acp_off_dyt, alphas_acp_off_dyt)}
    
    # Comparación
    print(f"\nCOMPARACIÓN LN vs DyT (gamma={g}):")
    comparison = pd.DataFrame({
        "Barra": df_acp_off_ln["Barra"],
        "Coverage_LN": df_acp_off_ln["Coverage"],
        "Coverage_DyT": df_acp_off_dyt["Coverage"],
        "Width_LN": df_acp_off_ln["Width"],
        "Width_DyT": df_acp_off_dyt["Width"],
        "MAE_LN": df_acp_off_ln["MAE"],
        "MAE_DyT": df_acp_off_dyt["MAE"],
        "Alpha_Mean_LN": df_acp_off_ln["Alpha_Mean"],
        "Alpha_Mean_DyT": df_acp_off_dyt["Alpha_Mean"]})
    display(comparison)


In [ ]:
# Graficamos
plot_cp(b, cp_acp_off_ln, cp_acp_off_dyt, resultados_stacking_ln, resultados_stacking_dyt, tipo_cp='ACP Offline')

# ACP Online

In [ ]:
from Modulos.Conformal_Prediction_Wrapper import acp_online_implementation
import time
# Reentrena el modelo en cada iteración 
# funcionamiento es similar al offline pero con la diferencia que el modelo se reentrena
#  en cada iteración, lo que permite que el modelo se adapte a los nuevos datos y errores 
# observados.
# Adpatatividad
# Reentrena

# LN
print(f"\nACP Online LN:")
estrategia_barras_ln = df_mejores_ln.set_index('Barra')['Mejor Estrategia'].to_dict()

tiempo_inicio_ln = time.time()
df_acp_online_ln, cp_acp_online_ln, alphas_acp_online_ln, tiempos_ln = acp_online_implementation(resultados_stacking_ln,
                                                                                                 b,
                                                                                                 estrategia_barras_ln,
                                                                                                 modelo_base='Ridge',
                                                                                                 alpha=0.05,
                                                                                                 gamma=0.05)
tiempo_total_ln = time.time() - tiempo_inicio_ln
display(df_acp_online_ln[['Barra', 'Estrategia', 'Coverage', 'Expected', 'Coverage_Diff', 'Width', 'MAE', 'RMSE', 'Alpha_Mean', 'Tiempo_Seg']])

# DyT
print(f"\nACP Online DyT:")
estrategia_barras_dyt = df_mejores_dyt.set_index('Barra')['Mejor Estrategia'].to_dict()

tiempo_inicio_dyt = time.time()
df_acp_online_dyt, cp_acp_online_dyt, alphas_acp_online_dyt, tiempos_dyt = acp_online_implementation(resultados_stacking_dyt,
                                                                                                     b,
                                                                                                     estrategia_barras_dyt,
                                                                                                     modelo_base='Ridge',
                                                                                                     alpha=0.05,
                                                                                                     gamma=0.05)
tiempo_total_dyt = time.time() - tiempo_inicio_dyt
display(df_acp_online_dyt[['Barra', 'Estrategia', 'Coverage', 'Expected', 'Coverage_Diff', 'Width', 'MAE', 'RMSE', 'Alpha_Mean', 'Tiempo_Seg']])

# Comparación LN vs DyT
print(f"\nCOMPARACIÓN LN vs DyT (ACP Online):")
comparison = pd.DataFrame({
    "Barra": df_acp_online_ln["Barra"],
    "Coverage_LN": df_acp_online_ln["Coverage"],
    "Coverage_DyT": df_acp_online_dyt["Coverage"],
    "Width_LN": df_acp_online_ln["Width"],
    "Width_DyT": df_acp_online_dyt["Width"],
    "MAE_LN": df_acp_online_ln["MAE"],
    "MAE_DyT": df_acp_online_dyt["MAE"],
    "RMSE_LN": df_acp_online_ln["RMSE"],
    "RMSE_DyT": df_acp_online_dyt["RMSE"],
    "Tiempo_LN": df_acp_online_ln["Tiempo_Seg"],
    "Tiempo_DyT": df_acp_online_dyt["Tiempo_Seg"]
})
display(comparison)

print("\nTIEMPOS DE EJECUCIÓN:")
print(f"  ACP Online LN:  {tiempo_total_ln:.2f}s")
print(f"  ACP Online DyT: {tiempo_total_dyt:.2f}s")
print(f"  TOTAL:          {tiempo_total_ln + tiempo_total_dyt:.2f}s")

In [ ]:
# Graficamos
plot_cp(b, cp_acp_online_ln, cp_acp_online_dyt, resultados_stacking_ln, resultados_stacking_dyt, tipo_cp='ACP Online')

# Comparativa CP

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Crear diccionario con todos los métodos (INCLUYENDO CP CLÁSICO)
todos_metodos = {
    'CP Clásico': {
        'ln': df_cp_ln,
        'dyt': df_cp_dyt,
        'cp_ln': cp_ln,
        'cp_dyt': cp_dyt,
        'color': '#FFD93D',
        'marker': 'D'
    },
    'EnbPI': {
        'ln': df_enbpi_ln,
        'dyt': df_enbpi_dyt,
        'cp_ln': cp_enbpi_ln,
        'cp_dyt': cp_enbpi_dyt,
        'color': '#FF6B6B',
        'marker': 'o'
    },
    'ACP Offline': {
        'ln': df_acp_off_ln,
        'dyt': df_acp_off_dyt,
        'cp_ln': cp_acp_off_ln,
        'cp_dyt': cp_acp_off_dyt,
        'color': '#4ECDC4',
        'marker': 's'
    },
    'ACP Online': {
        'ln': df_acp_online_ln,
        'dyt': df_acp_online_dyt,
        'cp_ln': cp_acp_online_ln,
        'cp_dyt': cp_acp_online_dyt,
        'color': '#45B7D1',
        'marker': '^'
    }
}

barras = ['ATACAMA', 'CARDONES', 'CHARRUA', 'CRUCERO']

# ============ 1. TABLA COMPARATIVA GENERAL ============

print("\n" + "="*150)
print("TABLA 1: COMPARATIVA GENERAL - TODOS LOS MÉTODOS CP (4 MÉTODOS)")
print("="*150 + "\n")

tabla_general = []

for metodo_nombre, metodo_data in todos_metodos.items():
    df_ln = metodo_data['ln']
    df_dyt = metodo_data['dyt']
    
    # Promedios por estrategia
    cov_ln = df_ln['Coverage'].mean()
    cov_dyt = df_dyt['Coverage'].mean()
    cov_promedio = (cov_ln + cov_dyt) / 2
    
    width_ln = df_ln['Width'].mean()
    width_dyt = df_dyt['Width'].mean()
    width_promedio = (width_ln + width_dyt) / 2
    
    mae_ln = df_ln['MAE'].mean()
    mae_dyt = df_dyt['MAE'].mean()
    mae_promedio = (mae_ln + mae_dyt) / 2
    
    rmse_ln = df_ln['RMSE'].mean()
    rmse_dyt = df_dyt['RMSE'].mean()
    rmse_promedio = (rmse_ln + rmse_dyt) / 2
    
    cov_error = abs(cov_promedio - 0.95)
    
    tabla_general.append({
        'Método': metodo_nombre,
        'Coverage_LN': f"{cov_ln:.4f}",
        'Coverage_DyT': f"{cov_dyt:.4f}",
        'Coverage_Promedio': f"{cov_promedio:.4f}",
        'Coverage_Error': f"{cov_error:.4f}",
        'Width_LN': f"{width_ln:.2f}",
        'Width_DyT': f"{width_dyt:.2f}",
        'Width_Promedio': f"{width_promedio:.2f}",
        'MAE_LN': f"{mae_ln:.4f}",
        'MAE_DyT': f"{mae_dyt:.4f}",
        'MAE_Promedio': f"{mae_promedio:.4f}",
        'RMSE_Promedio': f"{rmse_promedio:.4f}"
    })

df_tabla_general = pd.DataFrame(tabla_general)
display(df_tabla_general)

# ============ 2. TABLA POR BARRA ============

print("\n" + "="*200)
print("TABLA 2: COMPARATIVA POR BARRA - TODAS LAS BARRAS")
print("="*200 + "\n")

tabla_por_barra = []

for barra in barras:
    for metodo_nombre, metodo_data in todos_metodos.items():
        df_ln = metodo_data['ln']
        df_dyt = metodo_data['dyt']
        
        row_ln = df_ln[df_ln['Barra'] == barra].iloc[0] if len(df_ln[df_ln['Barra'] == barra]) > 0 else None
        row_dyt = df_dyt[df_dyt['Barra'] == barra].iloc[0] if len(df_dyt[df_dyt['Barra'] == barra]) > 0 else None
        
        if row_ln is not None and row_dyt is not None:
            cov_ln = row_ln['Coverage']
            cov_dyt = row_dyt['Coverage']
            cov_promedio = (cov_ln + cov_dyt) / 2
            
            width_ln = row_ln['Width']
            width_dyt = row_dyt['Width']
            width_promedio = (width_ln + width_dyt) / 2
            
            mae_ln = row_ln['MAE']
            mae_dyt = row_dyt['MAE']
            mae_promedio = (mae_ln + mae_dyt) / 2
            
            rmse_ln = row_ln['RMSE']
            rmse_dyt = row_dyt['RMSE']
            rmse_promedio = (rmse_ln + rmse_dyt) / 2
            
            tabla_por_barra.append({
                'Barra': barra,
                'Método': metodo_nombre,
                'Coverage_LN': f"{cov_ln:.4f}",
                'Coverage_DyT': f"{cov_dyt:.4f}",
                'Coverage_Promedio': f"{cov_promedio:.4f}",
                'Width_Promedio': f"{width_promedio:.2f}",
                'MAE_Promedio': f"{mae_promedio:.4f}",
                'RMSE_Promedio': f"{rmse_promedio:.4f}"
            })

df_tabla_por_barra = pd.DataFrame(tabla_por_barra)
display(df_tabla_por_barra)

# ============ 3. TABLA DE RANKINGS ============

print("\n" + "="*150)
print("TABLA 3: RANKINGS - MEJOR MÉTODO POR MÉTRICA")
print("="*150 + "\n")

# Coverage
print("COVERAGE (más alto = mejor):")
ranking_coverage = []
for barra in barras:
    mejores = []
    for metodo_nombre, metodo_data in todos_metodos.items():
        df_ln = metodo_data['ln']
        df_dyt = metodo_data['dyt']
        row_ln = df_ln[df_ln['Barra'] == barra].iloc[0] if len(df_ln[df_ln['Barra'] == barra]) > 0 else None
        row_dyt = df_dyt[df_dyt['Barra'] == barra].iloc[0] if len(df_dyt[df_dyt['Barra'] == barra]) > 0 else None
        
        if row_ln is not None and row_dyt is not None:
            cov_promedio = (row_ln['Coverage'] + row_dyt['Coverage']) / 2
            mejores.append((metodo_nombre, cov_promedio))
    
    mejores.sort(key=lambda x: x[1], reverse=True)
    ranking_coverage.append({
        'Barra': barra,
        '1er Lugar': f"{mejores[0][0]} ({mejores[0][1]:.4f})",
        '2do Lugar': f"{mejores[1][0]} ({mejores[1][1]:.4f})",
        '3er Lugar': f"{mejores[2][0]} ({mejores[2][1]:.4f})",
        '4to Lugar': f"{mejores[3][0]} ({mejores[3][1]:.4f})"
    })

df_ranking_coverage = pd.DataFrame(ranking_coverage)
print(df_ranking_coverage.to_string(index=False))

# Width (más bajo = mejor)
print("\n\nWIDTH (más bajo = mejor):")
ranking_width = []
for barra in barras:
    mejores = []
    for metodo_nombre, metodo_data in todos_metodos.items():
        df_ln = metodo_data['ln']
        df_dyt = metodo_data['dyt']
        row_ln = df_ln[df_ln['Barra'] == barra].iloc[0] if len(df_ln[df_ln['Barra'] == barra]) > 0 else None
        row_dyt = df_dyt[df_dyt['Barra'] == barra].iloc[0] if len(df_dyt[df_dyt['Barra'] == barra]) > 0 else None
        
        if row_ln is not None and row_dyt is not None:
            width_promedio = (row_ln['Width'] + row_dyt['Width']) / 2
            mejores.append((metodo_nombre, width_promedio))
    
    mejores.sort(key=lambda x: x[1])
    ranking_width.append({
        'Barra': barra,
        '1er Lugar': f"{mejores[0][0]} ({mejores[0][1]:.2f})",
        '2do Lugar': f"{mejores[1][0]} ({mejores[1][1]:.2f})",
        '3er Lugar': f"{mejores[2][0]} ({mejores[2][1]:.2f})",
        '4to Lugar': f"{mejores[3][0]} ({mejores[3][1]:.2f})"
    })

df_ranking_width = pd.DataFrame(ranking_width)
print(df_ranking_width.to_string(index=False))

# ============ 4. GRÁFICAS COMPARATIVAS ============

print("\n\n" + "="*150)
print("CREANDO GRÁFICAS COMPARATIVAS...")
print("="*150)

# Figura 1: Coverage por barra
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Comparativa Coverage - 4 Métodos CP', fontsize=16, fontweight='bold')

for idx, barra in enumerate(barras):
    ax = axes[idx // 2, idx % 2]
    
    coverages_metodos = []
    metodos_nombres = []
    colores = []
    
    for metodo_nombre, metodo_data in todos_metodos.items():
        df_ln = metodo_data['ln']
        df_dyt = metodo_data['dyt']
        
        row_ln = df_ln[df_ln['Barra'] == barra].iloc[0] if len(df_ln[df_ln['Barra'] == barra]) > 0 else None
        row_dyt = df_dyt[df_dyt['Barra'] == barra].iloc[0] if len(df_dyt[df_dyt['Barra'] == barra]) > 0 else None
        
        if row_ln is not None and row_dyt is not None:
            cov_promedio = (row_ln['Coverage'] + row_dyt['Coverage']) / 2
            coverages_metodos.append(cov_promedio)
            metodos_nombres.append(metodo_nombre)
            colores.append(metodo_data['color'])
    
    bars = ax.bar(metodos_nombres, coverages_metodos, color=colores, alpha=0.7, edgecolor='black', linewidth=2)
    ax.axhline(0.95, color='red', linestyle='--', linewidth=2, label='Objetivo (95%)')
    ax.set_ylabel('Coverage', fontsize=11, fontweight='bold')
    ax.set_title(f'{barra}', fontsize=12, fontweight='bold')
    ax.set_ylim([0.88, 0.96])
    ax.grid(axis='y', alpha=0.3)
    ax.legend()
    ax.tick_params(axis='x', rotation=45)
    
    # Agregar valores en barras
    for bar, val in zip(bars, coverages_metodos):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2%}', ha='center', va='bottom', fontweight='bold', fontsize=9)
plt.show()

# Figura 2: Width por barra
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Comparativa Width (Ancho de Banda) - 4 Métodos CP', fontsize=16, fontweight='bold')

for idx, barra in enumerate(barras):
    ax = axes[idx // 2, idx % 2]
    
    widths_metodos = []
    metodos_nombres = []
    colores = []
    
    for metodo_nombre, metodo_data in todos_metodos.items():
        df_ln = metodo_data['ln']
        df_dyt = metodo_data['dyt']
        
        row_ln = df_ln[df_ln['Barra'] == barra].iloc[0] if len(df_ln[df_ln['Barra'] == barra]) > 0 else None
        row_dyt = df_dyt[df_dyt['Barra'] == barra].iloc[0] if len(df_dyt[df_dyt['Barra'] == barra]) > 0 else None
        
        if row_ln is not None and row_dyt is not None:
            width_promedio = (row_ln['Width'] + row_dyt['Width']) / 2
            widths_metodos.append(width_promedio)
            metodos_nombres.append(metodo_nombre)
            colores.append(metodo_data['color'])
    
    bars = ax.bar(metodos_nombres, widths_metodos, color=colores, alpha=0.7, edgecolor='black', linewidth=2)
    ax.set_ylabel('Width (USD/MWh)', fontsize=11, fontweight='bold')
    ax.set_title(f'{barra}', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    
    # Agregar valores en barras
    for bar, val in zip(bars, widths_metodos):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
plt.show()

# Figura 3: MAE por barra
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Comparativa MAE - 4 Métodos CP', fontsize=16, fontweight='bold')

for idx, barra in enumerate(barras):
    ax = axes[idx // 2, idx % 2]
    
    maes_metodos = []
    metodos_nombres = []
    colores = []
    
    for metodo_nombre, metodo_data in todos_metodos.items():
        df_ln = metodo_data['ln']
        df_dyt = metodo_data['dyt']
        
        row_ln = df_ln[df_ln['Barra'] == barra].iloc[0] if len(df_ln[df_ln['Barra'] == barra]) > 0 else None
        row_dyt = df_dyt[df_dyt['Barra'] == barra].iloc[0] if len(df_dyt[df_dyt['Barra'] == barra]) > 0 else None
        
        if row_ln is not None and row_dyt is not None:
            mae_promedio = (row_ln['MAE'] + row_dyt['MAE']) / 2
            maes_metodos.append(mae_promedio)
            metodos_nombres.append(metodo_nombre)
            colores.append(metodo_data['color'])
    
    bars = ax.bar(metodos_nombres, maes_metodos, color=colores, alpha=0.7, edgecolor='black', linewidth=2)
    ax.set_ylabel('MAE (USD/MWh)', fontsize=11, fontweight='bold')
    ax.set_title(f'{barra}', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    
    # Agregar valores en barras
    for bar, val in zip(bars, maes_metodos):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
plt.show()

# Figura 4: Scatter Coverage vs Width
fig, ax = plt.subplots(figsize=(15, 9))

for metodo_nombre, metodo_data in todos_metodos.items():
    df_ln = metodo_data['ln']
    df_dyt = metodo_data['dyt']
    
    coverages = []
    widths = []
    
    for barra in barras:
        row_ln = df_ln[df_ln['Barra'] == barra].iloc[0] if len(df_ln[df_ln['Barra'] == barra]) > 0 else None
        row_dyt = df_dyt[df_dyt['Barra'] == barra].iloc[0] if len(df_dyt[df_dyt['Barra'] == barra]) > 0 else None
        
        if row_ln is not None and row_dyt is not None:
            cov_promedio = (row_ln['Coverage'] + row_dyt['Coverage']) / 2
            width_promedio = (row_ln['Width'] + row_dyt['Width']) / 2
            coverages.append(cov_promedio)
            widths.append(width_promedio)
    
    ax.scatter(widths, coverages, s=350, marker=metodo_data['marker'], 
               label=metodo_nombre, color=metodo_data['color'], alpha=0.7, edgecolors='black', linewidth=2)
    
    # Agregar etiquetas con nombres de barras
    for i, barra in enumerate(barras):
        ax.annotate(barra, (widths[i], coverages[i]), fontsize=8, ha='center', fontweight='bold')

ax.axhline(0.95, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Objetivo Coverage (95%)')
ax.set_xlabel('Width (USD/MWh)', fontsize=12, fontweight='bold')
ax.set_ylabel('Coverage', fontsize=12, fontweight='bold')
ax.set_title('Trade-off: Coverage vs Width - 4 Métodos CP', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='best', scatterpoints=1)
plt.show()

# Figura 5: Heatmap de Coverage
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# LN
data_heatmap_ln = []
for metodo_nombre, metodo_data in todos_metodos.items():
    row_data = []
    df_ln = metodo_data['ln']
    for barra in barras:
        row = df_ln[df_ln['Barra'] == barra]
        if len(row) > 0:
            row_data.append(row.iloc[0]['Coverage'])
        else:
            row_data.append(0)
    data_heatmap_ln.append(row_data)

sns.heatmap(data_heatmap_ln, annot=True, fmt='.4f', cmap='RdYlGn', center=0.95,
            xticklabels=barras, yticklabels=todos_metodos.keys(), ax=axes[0],
            cbar_kws={'label': 'Coverage'}, vmin=0.88, vmax=0.96)
axes[0].set_title('Heatmap Coverage - LN Strategy', fontsize=12, fontweight='bold')

# DyT
data_heatmap_dyt = []
for metodo_nombre, metodo_data in todos_metodos.items():
    row_data = []
    df_dyt = metodo_data['dyt']
    for barra in barras:
        row = df_dyt[df_dyt['Barra'] == barra]
        if len(row) > 0:
            row_data.append(row.iloc[0]['Coverage'])
        else:
            row_data.append(0)
    data_heatmap_dyt.append(row_data)

sns.heatmap(data_heatmap_dyt, annot=True, fmt='.4f', cmap='RdYlGn', center=0.95,
            xticklabels=barras, yticklabels=todos_metodos.keys(), ax=axes[1],
            cbar_kws={'label': 'Coverage'}, vmin=0.88, vmax=0.96)
axes[1].set_title('Heatmap Coverage - DyT Strategy', fontsize=12, fontweight='bold')
plt.show()

# ============ 5. TABLA RESUMEN EJECUTIVA ============

print("\n" + "="*180)
print("📋 TABLA 4: RESUMEN EJECUTIVO - CONCLUSIONES (4 MÉTODOS)")
print("="*180 + "\n")

tabla_resumen = pd.DataFrame({
    "Método": ["CP Clásico", "EnbPI", "ACP Offline", "ACP Online"],
    
    "Coverage Promedio": [
        f"{df_tabla_general.iloc[0]['Coverage_Promedio']}",
        f"{df_tabla_general.iloc[1]['Coverage_Promedio']}",
        f"{df_tabla_general.iloc[2]['Coverage_Promedio']}",
        f"{df_tabla_general.iloc[3]['Coverage_Promedio']}"
    ],
    
    "Coverage Error": [
        f"{df_tabla_general.iloc[0]['Coverage_Error']}",
        f"{df_tabla_general.iloc[1]['Coverage_Error']}",
        f"{df_tabla_general.iloc[2]['Coverage_Error']}",
        f"{df_tabla_general.iloc[3]['Coverage_Error']}"
    ],
    
    "Width Promedio": [
        f"{df_tabla_general.iloc[0]['Width_Promedio']} USD/MWh",
        f"{df_tabla_general.iloc[1]['Width_Promedio']} USD/MWh",
        f"{df_tabla_general.iloc[2]['Width_Promedio']} USD/MWh",
        f"{df_tabla_general.iloc[3]['Width_Promedio']} USD/MWh"
    ],
    
    "MAE Promedio": [
        f"{df_tabla_general.iloc[0]['MAE_Promedio']} USD/MWh",
        f"{df_tabla_general.iloc[1]['MAE_Promedio']} USD/MWh",
        f"{df_tabla_general.iloc[2]['MAE_Promedio']} USD/MWh",
        f"{df_tabla_general.iloc[3]['MAE_Promedio']} USD/MWh"
    ],
    
    "RMSE Promedio": [
        f"{df_tabla_general.iloc[0]['RMSE_Promedio']} USD/MWh",
        f"{df_tabla_general.iloc[1]['RMSE_Promedio']} USD/MWh",
        f"{df_tabla_general.iloc[2]['RMSE_Promedio']} USD/MWh",
        f"{df_tabla_general.iloc[3]['RMSE_Promedio']} USD/MWh"
    ],
    
    "Adaptativo": ["NO", "NO", "PARCIAL (α_t)", "SÍ (α_t + modelo)"],
    
    "Velocidad": ["⚡ < 1s", "⚡ < 1s", "⚡ ~2s", "⚡ ~5s"],
    
    "Recomendación": ["Baseline", "Baseline", "Muy Bueno", "⭐ PREFERIDO"]
})

print(tabla_resumen.to_string(index=False))

print("\n" + "="*180)
print("✅ COMPARATIVA CON 4 MÉTODOS COMPLETADA")
print("="*180)